<a href="https://colab.research.google.com/github/rasia92015-byte/Boltz2_colab/blob/main/notebooks/databases/PDBbind/PDBbind.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ======================================================================
# PDBbind v2020R1 COMPLETE WT/MT DISCOVERY PIPELINE
#
# INPUTS:
#   /content/drive/MyDrive/INDEX_general_PL.2020R1.lst
#   /content/drive/MyDrive/P-L.tar.gz
#
#
# ======================================================================


# ======================================================================
# 0. INSTALL
# ======================================================================

!pip -q install rdkit biopython requests openpyxl xlsxwriter


# ======================================================================
# 1. MOUNT DRIVE + IMPORT
# ======================================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import io
import re
import json
import math
import time
import tarfile
import shutil
from pathlib import Path
from collections import OrderedDict
from itertools import combinations
from urllib.parse import quote

import numpy as np
import pandas as pd
import requests

from rdkit import Chem
from rdkit.Chem.MolStandardize import rdMolStandardize

from Bio.Align import PairwiseAligner

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment
from openpyxl.utils import get_column_letter


# ======================================================================
# 2. PATHS
# ======================================================================

BASE = Path("/content/drive/MyDrive")

INDEX_FILE = BASE / "INDEX_general_PL.2020R1.lst"
PL_TAR     = BASE / "P-L.tar.gz"

OUT_XLSX = (
    BASE /
    "PDBbind_2020R1_singlechain_WT_MT_COMPLETE.xlsx"
)

MASTER_CSV = (
    BASE /
    "PDBbind_SingleChain_Master.csv.gz"
)

BOLTZ_CANDIDATE_CSV = (
    BASE /
    "PDBbind_Boltz_Unique_Input.csv"
)

BOLTZ_STRICT_CSV = (
    BASE /
    "PDBbind_Boltz_Strict_Input.csv"
)

RCSB_CACHE = (
    BASE /
    "PDBbind_RCSB_singlechain_cache_v3.jsonl"
)

UNIPROT_CACHE = (
    BASE /
    "PDBbind_UniProt_sequence_cache_v3.json"
)

CCD_CACHE = (
    BASE /
    "PDBbind_RCSB_CCD_name_cache.json"
)

PUBCHEM_CACHE = (
    BASE /
    "PDBbind_PubChem_name_cache.json"
)


# Try alternative capitalization automatically
if not PL_TAR.exists():

    alternatives = (
        list(BASE.glob("P-L*.tar.gz")) +
        list(BASE.glob("p-l*.tar.gz")) +
        list(BASE.glob("P_L*.tar.gz"))
    )

    if alternatives:
        PL_TAR = alternatives[0]


if not INDEX_FILE.exists():
    raise FileNotFoundError(
        f"Cannot find:\n{INDEX_FILE}"
    )

if not PL_TAR.exists():
    raise FileNotFoundError(
        "Cannot find P-L.tar.gz under:\n"
        f"{BASE}"
    )


print("INDEX:", INDEX_FILE)
print("P-L:  ", PL_TAR)
print("OUT:  ", OUT_XLSX)


# ======================================================================
# 3. PARSE PDBBIND GENERAL INDEX
# ======================================================================

def parse_general_index(path):

    rows = []

    pattern = re.compile(
        r"^"
        r"(\S+)\s+"              # PDB ID
        r"(\S+)\s+"              # resolution
        r"(\d{4})\s+"            # release year
        r"(\S+)\s+"              # affinity
        r"//\s+"
        r"(\S+)"                 # reference
        r"(?:\s+\(([^)]*)\))?"   # ligand descriptor
        r"\s*(.*)"
        r"$"
    )

    with open(
        path,
        "r",
        encoding="utf-8",
        errors="replace"
    ) as f:

        for line in f:

            line = line.rstrip("\n")

            if (
                not line.strip()
                or line.startswith("#")
            ):
                continue

            m = pattern.match(
                line.strip()
            )

            if not m:
                print(
                    "WARNING: unparsed index row:",
                    line
                )
                continue

            (
                pdb_id,
                resolution,
                year,
                affinity,
                reference,
                ligand_code,
                comment,
            ) = m.groups()

            rows.append({

                "PDB_ID":
                    pdb_id.upper(),

                "Resolution":
                    resolution,

                "Release_Year":
                    int(year),

                "Affinity_Raw":
                    affinity,

                "Affinity_Reference":
                    reference,

                "PDBbind_Ligand_Code":
                    ligand_code,

                "Index_Comment":
                    comment or "",

                "Index_Raw_Line":
                    line,
            })

    return pd.DataFrame(rows)


index_df = parse_general_index(
    INDEX_FILE
)


print("\nPDBbind rows:", len(index_df))
print(
    "Unique PDB IDs:",
    index_df["PDB_ID"].nunique()
)

assert len(index_df) > 18000, (
    "Too few index rows parsed. "
    "Check INDEX_general_PL.2020R1.lst."
)


# ======================================================================
# 4. PARSE AFFINITY
# ======================================================================

UNIT_TO_NM = {

    "fM": 1e-6,
    "pM": 1e-3,
    "nM": 1.0,
    "uM": 1e3,
    "mM": 1e6,
    "M":  1e9,
}


def parse_affinity(x):

    if pd.isna(x):

        return (
            None, None, None,
            None, None, None
        )

    x = str(x).strip()

    m = re.match(
        r"^(Kd|Ki|IC50)"
        r"(<=|>=|=|<|>|~)?"
        r"([0-9]*\.?[0-9]+)"
        r"(fM|pM|nM|uM|mM|M)$",
        x
    )

    if not m:

        return (
            None, None, None,
            None, None, None
        )

    kind = m.group(1)
    relation = m.group(2) or "="
    value = float(m.group(3))
    unit = m.group(4)

    value_nm = (
        value *
        UNIT_TO_NM[unit]
    )

    p_affinity = (
        9 -
        math.log10(value_nm)
        if value_nm > 0
        else None
    )

    return (
        kind,
        relation,
        value,
        unit,
        value_nm,
        p_affinity,
    )


parsed = (
    index_df[
        "Affinity_Raw"
    ]
    .apply(parse_affinity)
)


index_df[
    [
        "Affinity_Type",
        "Affinity_Relation",
        "Affinity_Value",
        "Affinity_Unit",
        "Affinity_nM",
        "pAffinity",
    ]
] = pd.DataFrame(
    parsed.tolist(),
    index=index_df.index
)


# ======================================================================
# 5. NORMALIZE REFERENCE
# ======================================================================

index_df[
    "Reference_Key"
] = (
    index_df[
        "Affinity_Reference"
    ]
    .astype("string")
    .str.strip()
    .str.lower()
)


# ======================================================================
# 6. COMMENT QC FLAGS
# ======================================================================

def get_comment_flags(x):

    x = str(
        x or ""
    ).lower()

    covalent = (
        "covalent" in x
    )

    incomplete_ligand = (
        "incomplete ligand" in x
    )

    different_protein = (
        "different protein" in x
    )

    incorrect_data = (
        "incorrect data" in x
    )

    xray_assay_mismatch = bool(
        re.search(
            r"x-ray.*while.*"
            r"(kd|ki|ic50)",
            x
        )
    )

    structure_assay_mismatch = any([

        different_protein,
        incorrect_data,
        xray_assay_mismatch,

        (
            "crystal is wildtype" in x
            and
            "affinity assay" in x
        ),

        (
            "mutation" in x
            and
            "binding assay" in x
            and
            "wild type" in x
        ),
    ])

    nonstandard = (
        "nonstandard assay" in x
    )

    return pd.Series({

        "Covalent_Flag":
            covalent,

        "Incomplete_Ligand_Flag":
            incomplete_ligand,

        "Assay_Structure_Mismatch_Flag":
            structure_assay_mismatch,

        "Nonstandard_Assay_Flag":
            nonstandard,
    })


comment_flags = (
    index_df[
        "Index_Comment"
    ]
    .apply(get_comment_flags)
)


index_df = pd.concat(
    [
        index_df,
        comment_flags
    ],
    axis=1
)


# ======================================================================
# 7. AMINO-ACID MAP FOR PDBbind protein.pdb
# ======================================================================

AA3_TO_1 = {

    "ALA":"A",
    "ARG":"R",
    "ASN":"N",
    "ASP":"D",
    "CYS":"C",
    "GLN":"Q",
    "GLU":"E",
    "GLY":"G",
    "HIS":"H",
    "ILE":"I",
    "LEU":"L",
    "LYS":"K",
    "MET":"M",
    "PHE":"F",
    "PRO":"P",
    "SER":"S",
    "THR":"T",
    "TRP":"W",
    "TYR":"Y",
    "VAL":"V",

    # common modified residues
    "MSE":"M",
    "SEC":"U",
    "PYL":"O",
    "SEP":"S",
    "TPO":"T",
    "PTR":"Y",
}


# ======================================================================
# 8. PARSE protein.pdb
#
# Requirement:
#
#     EXACTLY ONE protein chain
#
# This is literal chain count in PDBbind *_protein.pdb,
# not biological assembly oligomeric state.
# ======================================================================

def parse_protein_pdb_bytes(data):

    text = data.decode(
        "utf-8",
        errors="replace"
    )

    chains = OrderedDict()

    in_first_model = True
    model_seen = False


    for line in text.splitlines():

        if line.startswith("MODEL"):

            if model_seen:
                break

            model_seen = True
            in_first_model = True
            continue


        if (
            line.startswith("ENDMDL")
            and model_seen
        ):
            break


        if not line.startswith("ATOM  "):
            continue


        if len(line) < 27:
            continue


        chain_id = (
            line[21].strip()
            or "_"
        )

        resname = (
            line[17:20]
            .strip()
            .upper()
        )

        # only count recognized amino-acid residues
        if resname not in AA3_TO_1:
            continue


        resseq = line[22:26].strip()
        icode = line[26].strip()

        residue_key = (
            resseq,
            icode,
            resname
        )


        if chain_id not in chains:

            chains[
                chain_id
            ] = OrderedDict()


        if (
            residue_key
            not in chains[
                chain_id
            ]
        ):

            chains[
                chain_id
            ][
                residue_key
            ] = AA3_TO_1[
                resname
            ]


    chain_ids = list(
        chains.keys()
    )


    chain_sequences = {

        chain:
            "".join(
                residues.values()
            )

        for chain, residues
        in chains.items()
    }


    chain_lengths = {

        chain:
            len(sequence)

        for chain, sequence
        in chain_sequences.items()
    }


    count = len(chain_ids)


    if count == 1:

        single_chain_id = (
            chain_ids[0]
        )

        observed_sequence = (
            chain_sequences[
                single_chain_id
            ]
        )

    else:

        single_chain_id = None
        observed_sequence = None


    return {

        "Protein_Chain_Count":
            count,

        "Protein_Chain_IDs":
            ";".join(
                chain_ids
            ),

        "Protein_Chain_Lengths":
            ";".join(
                f"{c}:{chain_lengths[c]}"
                for c in chain_ids
            ),

        "Single_Protein_Chain":
            count == 1,

        "Single_Chain_ID":
            single_chain_id,

        "Observed_PDB_Sequence":
            observed_sequence,

        "Observed_PDB_Sequence_Length":
            (
                len(observed_sequence)
                if observed_sequence
                else None
            ),
    }


# ======================================================================
# 9. LIGAND STANDARDIZATION
# ======================================================================

def standardize_mol(mol):

    if mol is None:
        return None

    try:

        mol = (
            rdMolStandardize
            .Cleanup(mol)
        )

    except Exception:
        pass


    # use major molecular fragment to reduce salt differences
    try:

        mol = (
            rdMolStandardize
            .FragmentParent(mol)
        )

    except Exception:
        pass


    try:

        mol = Chem.RemoveHs(
            mol
        )

    except Exception:
        pass


    return mol


def mol_to_identifiers(mol):

    mol = standardize_mol(
        mol
    )

    if mol is None:
        return {}


    try:

        smiles = Chem.MolToSmiles(

            mol,

            canonical=True,

            isomericSmiles=True
        )

    except Exception:

        smiles = None


    try:

        inchikey = (
            Chem.MolToInchiKey(
                mol
            )
        )

    except Exception:

        inchikey = None


    if inchikey:

        inchikey14 = (
            inchikey.split("-")[0]
        )

    else:

        inchikey14 = None


    return {

        "Ligand_SMILES":
            smiles,

        "Ligand_InChIKey":
            inchikey,

        "Ligand_InChIKey14":
            inchikey14,
    }


def parse_sdf_bytes(data):

    try:

        supplier = (
            Chem.ForwardSDMolSupplier(

                io.BytesIO(data),

                sanitize=True,

                removeHs=True,

                strictParsing=False
            )
        )

        for mol in supplier:

            if mol is not None:

                return (
                    mol_to_identifiers(
                        mol
                    )
                )

    except Exception:
        pass


    return {}


def parse_mol2_bytes(data):

    try:

        text = data.decode(
            "utf-8",
            errors="replace"
        )

        mol = Chem.MolFromMol2Block(

            text,

            sanitize=True,

            removeHs=True
        )

        return (
            mol_to_identifiers(
                mol
            )
        )

    except Exception:

        return {}


# ======================================================================
# 10. STREAM P-L.tar.gz
#
# Does NOT unpack the entire archive.
# ======================================================================

wanted_pdbs = set(
    index_df[
        "PDB_ID"
    ]
)


structure_info = {

    pdb: {

        "PDB_ID":
            pdb,

        "Protein_File_Found":
            False,

        "Ligand_SDF_Found":
            False,

        "Ligand_MOL2_Found":
            False,

        "Protein_Chain_Count":
            None,

        "Protein_Chain_IDs":
            None,

        "Protein_Chain_Lengths":
            None,

        "Single_Protein_Chain":
            False,

        "Single_Chain_ID":
            None,

        "Observed_PDB_Sequence":
            None,

        "Observed_PDB_Sequence_Length":
            None,

        "Ligand_SMILES":
            None,

        "Ligand_InChIKey":
            None,

        "Ligand_InChIKey14":
            None,

        "Ligand_Parse_Source":
            None,
    }

    for pdb in wanted_pdbs
}


print("\nScanning P-L.tar.gz...")


relevant_files = 0


with tarfile.open(
    PL_TAR,
    mode="r|gz"
) as tf:

    for member in tf:

        if not member.isfile():
            continue


        filename = (
            os.path.basename(
                member.name
            )
            .lower()
        )


        m = re.match(
            r"^([0-9a-z]{4})_"
            r"(protein\.pdb|"
            r"ligand\.sdf|"
            r"ligand\.mol2)$",
            filename
        )


        if not m:
            continue


        pdb_id = (
            m.group(1)
            .upper()
        )

        kind = m.group(2)


        if pdb_id not in wanted_pdbs:
            continue


        fh = tf.extractfile(
            member
        )

        if fh is None:
            continue


        data = fh.read()

        rec = (
            structure_info[
                pdb_id
            ]
        )


        # ----------------------------------------------------------
        # PROTEIN
        # ----------------------------------------------------------

        if kind == "protein.pdb":

            rec[
                "Protein_File_Found"
            ] = True

            try:

                rec.update(
                    parse_protein_pdb_bytes(
                        data
                    )
                )

            except Exception as e:

                rec[
                    "Protein_Parse_Error"
                ] = repr(e)


        # ----------------------------------------------------------
        # SDF preferred
        # ----------------------------------------------------------

        elif kind == "ligand.sdf":

            rec[
                "Ligand_SDF_Found"
            ] = True

            ids = parse_sdf_bytes(
                data
            )

            if ids:

                rec.update(
                    ids
                )

                rec[
                    "Ligand_Parse_Source"
                ] = (
                    "PDBbind ligand.sdf"
                )


        # ----------------------------------------------------------
        # MOL2 fallback
        # ----------------------------------------------------------

        elif kind == "ligand.mol2":

            rec[
                "Ligand_MOL2_Found"
            ] = True

            if not rec.get(
                "Ligand_SMILES"
            ):

                ids = (
                    parse_mol2_bytes(
                        data
                    )
                )

                if ids:

                    rec.update(
                        ids
                    )

                    rec[
                        "Ligand_Parse_Source"
                    ] = (
                        "PDBbind ligand.mol2"
                    )


        relevant_files += 1


        if relevant_files % 5000 == 0:

            print(
                "Parsed archive files:",
                relevant_files
            )


structure_df = pd.DataFrame(
    structure_info.values()
)


# ======================================================================
# 11. MERGE INDEX + P-L
# ======================================================================

master = index_df.merge(

    structure_df,

    on="PDB_ID",

    how="left"
)


print(
    "\nTotal master rows:",
    len(master)
)

print(
    "Protein files:",
    master[
        "Protein_File_Found"
    ].fillna(False).sum()
)

print(
    "Exactly one protein chain:",
    (
        master[
            "Protein_Chain_Count"
        ] == 1
    ).sum()
)


# ======================================================================
# 12. KEEP EXACTLY ONE PROTEIN CHAIN
# ======================================================================

single_chain = (

    master[
        master[
            "Protein_Chain_Count"
        ] == 1
    ]

    .copy()

    .reset_index(
        drop=True
    )
)


print(
    "\nSingle-chain structures:",
    len(single_chain)
)


# ======================================================================
# 13. REQUEST SESSION
# ======================================================================

retry = Retry(

    total=6,

    connect=6,

    read=6,

    backoff_factor=1,

    status_forcelist=[
        429,
        500,
        502,
        503,
        504,
    ],

    allowed_methods=[
        "GET",
        "POST"
    ]
)


session = requests.Session()

session.mount(

    "https://",

    HTTPAdapter(
        max_retries=retry
    )
)


RCSB_GRAPHQL = (
    "https://data.rcsb.org/graphql"
)


# ======================================================================
# 14. RCSB PROTEIN ENTITY QUERY
# ======================================================================

RCSB_PROTEIN_QUERY = """
query GetEntries($ids: [String!]!) {

  entries(entry_ids: $ids) {

    rcsb_id

    polymer_entities {

      rcsb_id

      entity_poly {

        rcsb_entity_polymer_type
        pdbx_seq_one_letter_code_can
        rcsb_mutation_count

      }

      rcsb_polymer_entity {

        pdbx_description

      }

      rcsb_polymer_entity_container_identifiers {

        auth_asym_ids

        reference_sequence_identifiers {

          database_name
          database_accession

        }

      }

    }

  }

}
"""


def rcsb_fetch_entries(ids):

    payload = {

        "query":
            RCSB_PROTEIN_QUERY,

        "variables": {
            "ids": ids
        }
    }


    r = session.post(

        RCSB_GRAPHQL,

        json=payload,

        timeout=120
    )

    r.raise_for_status()

    data = r.json()


    if data.get("errors"):

        raise RuntimeError(
            str(data["errors"])
        )


    return (
        data
        .get("data", {})
        .get("entries", [])
        or []
    )


# ======================================================================
# 15. RCSB HELPERS
# ======================================================================

def clean_sequence(seq):

    if not seq:
        return None

    seq = (
        str(seq)
        .upper()
    )

    seq = re.sub(
        r"[^A-Z]",
        "",
        seq
    )

    return (
        seq
        if seq
        else None
    )


def get_uniprot_ids(entity):

    container = (

        entity.get(
            "rcsb_polymer_entity_container_identifiers"
        )

        or {}
    )


    refs = (

        container.get(
            "reference_sequence_identifiers"
        )

        or []
    )


    ids = []


    for ref in refs:

        database = str(
            ref.get(
                "database_name"
            )
            or ""
        ).upper()

        accession = (
            ref.get(
                "database_accession"
            )
        )


        if (
            "UNIPROT" in database
            and accession
        ):

            ids.append(
                str(accession)
            )


    return list(
        dict.fromkeys(ids)
    )


def select_protein_entity(
    entry,
    pdbbind_chain
):

    if not entry:

        return (
            None,
            "ENTRY_MISSING"
        )


    entities = (

        entry.get(
            "polymer_entities"
        )

        or []
    )


    proteins = []


    for entity in entities:

        entity_poly = (

            entity.get(
                "entity_poly"
            )

            or {}
        )


        polymer_type = str(
            entity_poly.get(
                "rcsb_entity_polymer_type"
            )
            or ""
        ).lower()


        if polymer_type == "protein":

            proteins.append(
                entity
            )


    if len(proteins) == 0:

        return (
            None,
            "NO_PROTEIN_ENTITY"
        )


    # Exact chain match first
    matches = []


    if pdbbind_chain:

        for entity in proteins:

            container = (

                entity.get(
                    "rcsb_polymer_entity_container_identifiers"
                )

                or {}
            )


            auth_chains = [

                str(x)

                for x in (

                    container.get(
                        "auth_asym_ids"
                    )

                    or []
                )
            ]


            if (
                str(pdbbind_chain)
                in auth_chains
            ):

                matches.append(
                    entity
                )


    if len(matches) == 1:

        return (
            matches[0],
            "CHAIN_MATCH"
        )


    # If RCSB only has one protein entity
    if len(proteins) == 1:

        return (
            proteins[0],
            "ONLY_PROTEIN_ENTITY"
        )


    return (
        None,
        "AMBIGUOUS_ENTITY"
    )


def parse_rcsb_entry(
    entry,
    pdb_id,
    pdbbind_chain
):

    result = {
        "PDB_ID": pdb_id
    }


    entity, status = (

        select_protein_entity(

            entry,

            pdbbind_chain
        )
    )


    result[
        "RCSB_Entity_Select_Status"
    ] = status


    if entity is None:

        result.update({

            "UniProt_ID":
                None,

            "UniProt_ID_Count":
                0,

            "RCSB_Target_Name":
                None,

            "Sequence":
                None,

            "Sequence_Length":
                None,

            "RCSB_Mutation_Count":
                None,
        })

        return result


    entity_poly = (

        entity.get(
            "entity_poly"
        )

        or {}
    )


    polymer_info = (

        entity.get(
            "rcsb_polymer_entity"
        )

        or {}
    )


    ids = get_uniprot_ids(
        entity
    )


    sequence = clean_sequence(

        entity_poly.get(
            "pdbx_seq_one_letter_code_can"
        )
    )


    mutation_count = (

        entity_poly.get(
            "rcsb_mutation_count"
        )
    )


    try:

        mutation_count = int(
            mutation_count
        )

    except Exception:

        mutation_count = None


    result.update({

        "RCSB_Entity_ID":
            entity.get(
                "rcsb_id"
            ),

        "UniProt_ID":
            (
                ";".join(ids)
                if ids
                else None
            ),

        "UniProt_ID_Count":
            len(ids),

        "RCSB_Target_Name":
            polymer_info.get(
                "pdbx_description"
            ),

        "Sequence":
            sequence,

        "Sequence_Length":
            (
                len(sequence)
                if sequence
                else None
            ),

        "RCSB_Mutation_Count":
            mutation_count,
    })


    return result


# ======================================================================
# 16. LOAD VALID RCSB CACHE
# ======================================================================

rcsb_cache = {}


if RCSB_CACHE.exists():

    with open(
        RCSB_CACHE,
        "r"
    ) as f:

        for line in f:

            try:

                rec = json.loads(
                    line
                )

            except Exception:
                continue


            pdb_id = rec.get(
                "PDB_ID"
            )


            # only trust useful cache records
            if (
                pdb_id
                and
                "RCSB_Entity_Select_Status"
                in rec
                and
                "UniProt_ID_Count"
                in rec
            ):

                rcsb_cache[
                    pdb_id
                ] = rec


print(
    "\nUsable RCSB cache:",
    len(rcsb_cache)
)


# ======================================================================
# 17. RCSB BATCH PROCESS
# ======================================================================

single_chain_ids = sorted(
    single_chain[
        "PDB_ID"
    ].unique()
)


chain_lookup = dict(
    zip(

        single_chain[
            "PDB_ID"
        ],

        single_chain[
            "Single_Chain_ID"
        ]
    )
)


todo = [

    x for x in single_chain_ids

    if x not in rcsb_cache
]


print(
    "RCSB structures to retrieve:",
    len(todo)
)


def save_rcsb_record(rec):

    pdb_id = rec[
        "PDB_ID"
    ]

    rcsb_cache[
        pdb_id
    ] = rec


    with open(
        RCSB_CACHE,
        "a"
    ) as f:

        f.write(
            json.dumps(
                rec
            )
            + "\n"
        )


def process_rcsb_batch(ids):

    if not ids:
        return


    try:

        entries = (
            rcsb_fetch_entries(
                ids
            )
        )


        lookup = {

            str(
                e.get(
                    "rcsb_id"
                )
            ).upper():
                e

            for e in entries

            if e
        }


        for pdb_id in ids:

            rec = parse_rcsb_entry(

                lookup.get(
                    pdb_id
                ),

                pdb_id,

                chain_lookup.get(
                    pdb_id
                )
            )

            save_rcsb_record(
                rec
            )


    except Exception as e:

        # split large failed requests
        if len(ids) > 1:

            midpoint = (
                len(ids) // 2
            )

            process_rcsb_batch(
                ids[:midpoint]
            )

            process_rcsb_batch(
                ids[midpoint:]
            )

        else:

            pdb_id = ids[0]

            save_rcsb_record({

                "PDB_ID":
                    pdb_id,

                "RCSB_Entity_Select_Status":
                    "API_ERROR",

                "UniProt_ID":
                    None,

                "UniProt_ID_Count":
                    0,

                "RCSB_Target_Name":
                    None,

                "Sequence":
                    None,

                "Sequence_Length":
                    None,

                "RCSB_Mutation_Count":
                    None,

                "RCSB_API_Error":
                    repr(e),
            })


BATCH_SIZE = 75


for start in range(
    0,
    len(todo),
    BATCH_SIZE
):

    batch = todo[
        start:
        start + BATCH_SIZE
    ]

    process_rcsb_batch(
        batch
    )


    done = min(
        start + BATCH_SIZE,
        len(todo)
    )


    if (
        done % 750 == 0
        or
        done == len(todo)
    ):

        print(
            f"RCSB: "
            f"{done}/{len(todo)}"
        )


    time.sleep(
        0.1
    )


# ======================================================================
# 18. MERGE RCSB DATA
# ======================================================================

rcsb_df = pd.DataFrame(
    rcsb_cache.values()
)


single_chain = (
    single_chain
    .merge(

        rcsb_df,

        on="PDB_ID",

        how="left",

        validate="one_to_one"
    )
)


print(
    "\nSingle-chain master:",
    len(single_chain)
)

print(
    "Exactly 1 UniProt:",
    (
        single_chain[
            "UniProt_ID_Count"
        ] == 1
    ).sum()
)


# ======================================================================
# 19. LIGAND IDENTITY KEY
#
# Connectivity InChIKey is used for candidate grouping.
#
# Full InChIKey is retained for stricter QC.
# ======================================================================

def ligand_group_key(row):

    ik14 = row.get(
        "Ligand_InChIKey14"
    )

    if (
        pd.notna(ik14)
        and str(ik14).strip()
    ):

        return (
            "IK14:"
            +
            str(ik14)
        )


    smiles = row.get(
        "Ligand_SMILES"
    )

    if (
        pd.notna(smiles)
        and str(smiles).strip()
    ):

        return (
            "SMILES:"
            +
            str(smiles)
        )


    return None


single_chain[
    "Ligand_Group_Key"
] = single_chain.apply(

    ligand_group_key,

    axis=1
)


# ======================================================================
# 20. RCSB CCD LIGAND NAMES — BATCH
# ======================================================================

RCSB_CCD_QUERY = """
query GetChemComps($ids: [String!]!) {

  chem_comps(comp_ids: $ids) {

    rcsb_id

    chem_comp {
      name
      formula
      type
    }

  }

}
"""


def load_json_cache(path):

    if not path.exists():
        return {}

    try:

        with open(
            path,
            "r"
        ) as f:

            return json.load(f)

    except Exception:

        return {}


def save_json_cache(
    path,
    obj
):

    with open(
        path,
        "w"
    ) as f:

        json.dump(
            obj,
            f,
            indent=2
        )


ccd_cache = load_json_cache(
    CCD_CACHE
)


ccd_codes = sorted(
    set(

        str(x).strip().upper()

        for x in single_chain[
            "PDBbind_Ligand_Code"
        ].dropna()

        if re.fullmatch(
            r"[A-Za-z0-9]{1,5}",
            str(x).strip()
        )
    )
)


ccd_todo = [

    x for x in ccd_codes

    if x not in ccd_cache
]


def fetch_ccd_batch(ids):

    payload = {

        "query":
            RCSB_CCD_QUERY,

        "variables": {
            "ids": ids
        }
    }


    r = session.post(

        RCSB_GRAPHQL,

        json=payload,

        timeout=120
    )

    r.raise_for_status()

    obj = r.json()


    if obj.get("errors"):

        raise RuntimeError(
            str(obj["errors"])
        )


    return (

        obj
        .get("data", {})
        .get("chem_comps", [])

        or []
    )


for start in range(
    0,
    len(ccd_todo),
    200
):

    batch = (
        ccd_todo[
            start:
            start + 200
        ]
    )


    try:

        results = (
            fetch_ccd_batch(
                batch
            )
        )


        found = set()


        for result in results:

            if not result:
                continue

            code = str(
                result.get(
                    "rcsb_id"
                )
            ).upper()

            chem = (
                result.get(
                    "chem_comp"
                )
                or {}
            )

            ccd_cache[
                code
            ] = {

                "name":
                    chem.get(
                        "name"
                    ),

                "formula":
                    chem.get(
                        "formula"
                    ),

                "type":
                    chem.get(
                        "type"
                    ),
            }

            found.add(
                code
            )


        for code in batch:

            if code not in found:

                ccd_cache[
                    code
                ] = {
                    "name": None
                }


    except Exception:

        # mark unresolved, PubChem fallback can still run
        for code in batch:

            ccd_cache.setdefault(

                code,

                {
                    "name":
                        None
                }
            )


save_json_cache(
    CCD_CACHE,
    ccd_cache
)


# ======================================================================
# 21. PUBCHEM TITLE FALLBACK
#
# Only used when CCD name unavailable.
# ======================================================================

pubchem_cache = load_json_cache(
    PUBCHEM_CACHE
)


def get_pubchem_title(
    inchikey
):

    if not inchikey:
        return None


    if inchikey in pubchem_cache:

        return (
            pubchem_cache[
                inchikey
            ]
        )


    url = (

        "https://pubchem.ncbi.nlm.nih.gov/"
        "rest/pug/compound/inchikey/"

        +
        quote(
            inchikey,
            safe=""
        )

        +
        "/property/Title/JSON"
    )


    title = None


    try:

        r = session.get(

            url,

            timeout=30
        )

        if r.status_code == 200:

            data = r.json()

            props = (

                data
                .get(
                    "PropertyTable",
                    {}
                )
                .get(
                    "Properties",
                    []
                )
            )


            if props:

                title = (
                    props[0]
                    .get(
                        "Title"
                    )
                )


    except Exception:
        pass


    pubchem_cache[
        inchikey
    ] = title


    time.sleep(
        0.05
    )


    return title


# ======================================================================
# 22. ADD HUMAN-READABLE LIGAND NAME
#
# Preference:
#
#   PubChem common/title name
#       ↓
#   RCSB CCD name
#       ↓
#   descriptive PDBbind field
# ======================================================================

unique_ligands = (

    single_chain[
        [
            "PDBbind_Ligand_Code",
            "Ligand_InChIKey"
        ]
    ]

    .drop_duplicates()

    .copy()
)


ligand_name_lookup = {}


for i, row in unique_ligands.iterrows():

    code = row[
        "PDBbind_Ligand_Code"
    ]

    inchikey = row[
        "Ligand_InChIKey"
    ]


    code_str = (
        str(code).strip()
        if pd.notna(code)
        else None
    )


    ik_str = (
        str(inchikey).strip()
        if pd.notna(inchikey)
        else None
    )


    ccd_name = None


    if (
        code_str
        and
        re.fullmatch(
            r"[A-Za-z0-9]{1,5}",
            code_str
        )
    ):

        ccd_name = (

            ccd_cache
            .get(
                code_str.upper(),
                {}
            )
            .get(
                "name"
            )
        )


    pubchem_name = (
        get_pubchem_title(
            ik_str
        )
        if ik_str
        else None
    )


    if pubchem_name:

        final_name = (
            pubchem_name
        )

        name_source = (
            "PubChem title"
        )


    elif ccd_name:

        final_name = (
            ccd_name
        )

        name_source = (
            "RCSB CCD"
        )


    elif (
        code_str
        and
        re.fullmatch(
            r"\d+-mer",
            code_str,
            flags=re.I
        )
    ):

        final_name = (
            f"{code_str} "
            "peptide/oligomer ligand"
        )

        name_source = (
            "PDBbind descriptor"
        )


    else:

        final_name = (
            code_str
            if code_str
            else "Unknown ligand"
        )

        name_source = (
            "PDBbind descriptor"
        )


    ligand_name_lookup[
        (
            code_str,
            ik_str
        )
    ] = {

        "Ligand_Name":
            final_name,

        "Ligand_CCD_Name":
            ccd_name,

        "Ligand_PubChem_Name":
            pubchem_name,

        "Ligand_Name_Source":
            name_source,
    }


save_json_cache(
    PUBCHEM_CACHE,
    pubchem_cache
)


def add_ligand_name(row):

    code = (
        str(
            row.get(
                "PDBbind_Ligand_Code"
            )
        ).strip()

        if pd.notna(
            row.get(
                "PDBbind_Ligand_Code"
            )
        )

        else None
    )


    ik = (
        str(
            row.get(
                "Ligand_InChIKey"
            )
        ).strip()

        if pd.notna(
            row.get(
                "Ligand_InChIKey"
            )
        )

        else None
    )


    return pd.Series(

        ligand_name_lookup.get(

            (
                code,
                ik
            ),

            {
                "Ligand_Name":
                    code,

                "Ligand_CCD_Name":
                    None,

                "Ligand_PubChem_Name":
                    None,

                "Ligand_Name_Source":
                    "PDBbind descriptor",
            }
        )
    )


ligand_annotations = (
    single_chain.apply(
        add_ligand_name,
        axis=1
    )
)


single_chain = pd.concat(
    [
        single_chain,
        ligand_annotations
    ],
    axis=1
)


# ======================================================================
# 23. PRELIMINARY GROUP INPUT
#
# HARD GROUP DEFINITION:
#
#   same UniProt
#   same ligand connectivity
#   same reference
#   same Kd/Ki/IC50
#
# ======================================================================

pre_pair = single_chain[

    (
        single_chain[
            "Protein_Chain_Count"
        ] == 1
    )

    &

    (
        single_chain[
            "UniProt_ID_Count"
        ] == 1
    )

    &

    (
        single_chain[
            "Ligand_Group_Key"
        ].notna()
    )

    &

    (
        single_chain[
            "Reference_Key"
        ].notna()
    )

    &

    (
        single_chain[
            "Affinity_Type"
        ].isin(
            [
                "Kd",
                "Ki",
                "IC50"
            ]
        )
    )

    &

    (
        single_chain[
            "Sequence"
        ].notna()
    )

].copy()


GROUP_COLS = [

    "UniProt_ID",
    "Ligand_Group_Key",
    "Reference_Key",
    "Affinity_Type",
]


# Only groups containing >= 2 structures can possibly make a pair
group_sizes = (

    pre_pair
    .groupby(
        GROUP_COLS,
        dropna=False
    )
    .size()
)


multi_groups = (
    group_sizes[
        group_sizes >= 2
    ]
    .index
)


multi_group_set = set(
    multi_groups.tolist()
)


pre_pair[
    "_group_tuple"
] = list(
    zip(
        *[
            pre_pair[c]
            for c in GROUP_COLS
        ]
    )
)


pair_input = (

    pre_pair[
        pre_pair[
            "_group_tuple"
        ].isin(
            multi_group_set
        )
    ]

    .copy()
)


print(
    "\nRows in potential multi-structure groups:",
    len(pair_input)
)


# ======================================================================
# 24. UNIPROT CANONICAL SEQUENCE CACHE
# ======================================================================

uniprot_cache = load_json_cache(
    UNIPROT_CACHE
)


needed_uniprots = sorted(

    set(

        str(x).split(";")[0]

        for x in pair_input[
            "UniProt_ID"
        ].dropna()

        if str(x).strip()
    )
)


print(
    "UniProt sequences needed:",
    len(needed_uniprots)
)


def fetch_uniprot_sequence(
    accession
):

    if accession in uniprot_cache:

        return (
            uniprot_cache[
                accession
            ]
        )


    url = (

        "https://rest.uniprot.org/"
        f"uniprotkb/{accession}.fasta"
    )


    sequence = None


    try:

        r = session.get(

            url,

            timeout=60
        )

        r.raise_for_status()


        sequence = "".join(

            line.strip()

            for line in r.text.splitlines()

            if (
                line.strip()
                and
                not line.startswith(">")
            )
        ).upper()


        sequence = re.sub(
            r"[^A-Z]",
            "",
            sequence
        )


        if not sequence:

            sequence = None


    except Exception as e:

        print(
            "UniProt failed:",
            accession,
            repr(e)
        )


    uniprot_cache[
        accession
    ] = sequence


    return sequence


for i, accession in enumerate(
    needed_uniprots,
    start=1
):

    fetch_uniprot_sequence(
        accession
    )


    if (
        i % 100 == 0
        or
        i == len(
            needed_uniprots
        )
    ):

        print(
            f"UniProt: "
            f"{i}/"
            f"{len(needed_uniprots)}"
        )


    time.sleep(
        0.02
    )


save_json_cache(
    UNIPROT_CACHE,
    uniprot_cache
)


# ======================================================================
# 25. ALIGN PDB CONSTRUCT TO UNIPROT
# ======================================================================

aligner = PairwiseAligner()

aligner.mode = "local"

aligner.match_score = 2.0
aligner.mismatch_score = -1.0

aligner.open_gap_score = -10.0
aligner.extend_gap_score = -0.5


VALID_AA = set(
    "ACDEFGHIKLMNPQRSTVWY"
)


def sequence_alignment_annotation(
    reference,
    construct
):

    reference = clean_sequence(
        reference
    )

    construct = clean_sequence(
        construct
    )


    if (
        not reference
        or
        not construct
    ):

        return {

            "Mapped_Positions":
                set(),

            "Residue_Map":
                {},

            "Substitutions":
                set(),

            "Alignment_Coverage":
                None,

            "Alignment_Identity":
                None,

            "UniProt_Map_Start":
                None,

            "UniProt_Map_End":
                None,
        }


    try:

        alignments = (
            aligner.align(
                reference,
                construct
            )
        )

        if len(
            alignments
        ) == 0:

            raise ValueError(
                "No alignment"
            )


        aln = (
            alignments[0]
        )


    except Exception:

        return {

            "Mapped_Positions":
                set(),

            "Residue_Map":
                {},

            "Substitutions":
                set(),

            "Alignment_Coverage":
                0,

            "Alignment_Identity":
                0,

            "UniProt_Map_Start":
                None,

            "UniProt_Map_End":
                None,
        }


    ref_blocks = (
        aln.aligned[0]
    )

    query_blocks = (
        aln.aligned[1]
    )


    residue_map = {}


    for (
        ref_block,
        query_block
    ) in zip(
        ref_blocks,
        query_blocks
    ):

        r0, r1 = map(
            int,
            ref_block
        )

        q0, q1 = map(
            int,
            query_block
        )


        length = min(
            r1-r0,
            q1-q0
        )


        for offset in range(
            length
        ):

            ref_i = (
                r0 + offset
            )

            q_i = (
                q0 + offset
            )


            if (
                ref_i < len(reference)
                and
                q_i < len(construct)
            ):

                residue_map[
                    ref_i + 1
                ] = construct[
                    q_i
                ]


    mapped_positions = set(
        residue_map.keys()
    )


    if not mapped_positions:

        return {

            "Mapped_Positions":
                set(),

            "Residue_Map":
                {},

            "Substitutions":
                set(),

            "Alignment_Coverage":
                0,

            "Alignment_Identity":
                0,

            "UniProt_Map_Start":
                None,

            "UniProt_Map_End":
                None,
        }


    substitutions = set()

    identical = 0


    for pos, observed_aa in (
        residue_map.items()
    ):

        ref_aa = (
            reference[
                pos - 1
            ]
        )


        if (
            observed_aa == ref_aa
        ):

            identical += 1


        elif (
            observed_aa in VALID_AA
            and
            ref_aa in VALID_AA
        ):

            substitutions.add(

                f"{ref_aa}"
                f"{pos}"
                f"{observed_aa}"
            )


    coverage = (

        len(mapped_positions)
        /
        len(construct)
    )


    identity = (

        identical
        /
        len(mapped_positions)
    )


    return {

        "Mapped_Positions":
            mapped_positions,

        "Residue_Map":
            residue_map,

        "Substitutions":
            substitutions,

        "Alignment_Coverage":
            coverage,

        "Alignment_Identity":
            identity,

        "UniProt_Map_Start":
            min(
                mapped_positions
            ),

        "UniProt_Map_End":
            max(
                mapped_positions
            ),
    }


# ======================================================================
# 26. ANNOTATE EACH POTENTIAL PAIR STRUCTURE
# ======================================================================

seq_annotation = {}


for i, row in pair_input.iterrows():

    pdb_id = row[
        "PDB_ID"
    ]

    accession = (

        str(
            row[
                "UniProt_ID"
            ]
        )

        .split(";")[0]
    )


    ref_seq = (
        uniprot_cache.get(
            accession
        )
    )


    ann = (
        sequence_alignment_annotation(

            ref_seq,

            row[
                "Sequence"
            ]
        )
    )


    seq_annotation[
        pdb_id
    ] = ann


# Add readable sequence annotation columns
pair_input[
    "Substitutions_vs_UniProt"
] = pair_input[
    "PDB_ID"
].map(

    lambda x:

        ";".join(
            sorted(
                seq_annotation[
                    x
                ][
                    "Substitutions"
                ]
            )
        )
)


pair_input[
    "Substitution_Count_vs_UniProt"
] = pair_input[
    "PDB_ID"
].map(

    lambda x:

        len(
            seq_annotation[
                x
            ][
                "Substitutions"
            ]
        )
)


pair_input[
    "UniProt_Alignment_Coverage"
] = pair_input[
    "PDB_ID"
].map(

    lambda x:

        seq_annotation[
            x
        ][
            "Alignment_Coverage"
        ]
)


pair_input[
    "UniProt_Alignment_Identity"
] = pair_input[
    "PDB_ID"
].map(

    lambda x:

        seq_annotation[
            x
        ][
            "Alignment_Identity"
        ]
)


pair_input[
    "UniProt_Map_Start"
] = pair_input[
    "PDB_ID"
].map(

    lambda x:

        seq_annotation[
            x
        ][
            "UniProt_Map_Start"
        ]
)


pair_input[
    "UniProt_Map_End"
] = pair_input[
    "PDB_ID"
].map(

    lambda x:

        seq_annotation[
            x
        ][
            "UniProt_Map_End"
        ]
)


# ======================================================================
# 27. WT/MT PAIR DISCOVERY
#
# Within each same-target/same-ligand/same-ref/same-endpoint group:
#
# A can be reference/WT and B mutant if
#
#     substitutions(A) is a proper subset of substitutions(B)
#
# Example:
#
# A = {A100V}
# B = {A100V, D816H}
#
# pair mutation = D816H
#
# This is much better than demanding mutation_count(A)==0.
# ======================================================================

candidate_rows = []

group_counter = 0


for group_key, group in (

    pair_input
    .groupby(
        GROUP_COLS,
        dropna=False,
        sort=False
    )
):

    if len(group) < 2:
        continue


    group_counter += 1

    group_id = (
        f"WTMT_{group_counter:05d}"
    )


    records = list(
        group.to_dict(
            "records"
        )
    )


    for a, b in combinations(
        records,
        2
    ):

        a_pdb = a[
            "PDB_ID"
        ]

        b_pdb = b[
            "PDB_ID"
        ]


        a_ann = (
            seq_annotation[
                a_pdb
            ]
        )

        b_ann = (
            seq_annotation[
                b_pdb
            ]
        )


        a_sub = (
            a_ann[
                "Substitutions"
            ]
        )

        b_sub = (
            b_ann[
                "Substitutions"
            ]
        )


        # ----------------------------------------------------------
        # Establish direction:
        # smaller substitution set = reference/WT-like
        # larger proper superset = mutant
        # ----------------------------------------------------------

        if a_sub < b_sub:

            wt = a
            mt = b

            wt_ann = a_ann
            mt_ann = b_ann


        elif b_sub < a_sub:

            wt = b
            mt = a

            wt_ann = b_ann
            mt_ann = a_ann


        else:

            # no clear reference -> mutant relationship
            continue


        pair_specific = (

            mt_ann[
                "Substitutions"
            ]

            -

            wt_ann[
                "Substitutions"
            ]
        )


        if not pair_specific:
            continue


        # ----------------------------------------------------------
        # Shared canonical target-region overlap
        # ----------------------------------------------------------

        wt_positions = (
            wt_ann[
                "Mapped_Positions"
            ]
        )

        mt_positions = (
            mt_ann[
                "Mapped_Positions"
            ]
        )


        if (
            wt_positions
            and
            mt_positions
        ):

            mapped_overlap = (

                len(
                    wt_positions
                    &
                    mt_positions
                )

                /

                min(
                    len(
                        wt_positions
                    ),
                    len(
                        mt_positions
                    )
                )
            )

        else:

            mapped_overlap = 0


        # ----------------------------------------------------------
        # Direct residue differences at positions present in both
        # ----------------------------------------------------------

        direct_diffs = []


        for pos in sorted(
            wt_positions
            &
            mt_positions
        ):

            wt_aa = (
                wt_ann[
                    "Residue_Map"
                ][pos]
            )

            mt_aa = (
                mt_ann[
                    "Residue_Map"
                ][pos]
            )


            if (
                wt_aa != mt_aa
                and
                wt_aa in VALID_AA
                and
                mt_aa in VALID_AA
            ):

                direct_diffs.append(

                    f"{wt_aa}"
                    f"{pos}"
                    f"{mt_aa}"
                )


        mutation = (
            ";".join(
                sorted(
                    pair_specific
                )
            )
        )


        wt_aff = wt.get(
            "Affinity_nM"
        )

        mt_aff = mt.get(
            "Affinity_nM"
        )


        if (
            pd.notna(wt_aff)
            and
            pd.notna(mt_aff)
            and
            wt_aff > 0
            and
            mt_aff > 0
        ):

            delta_log = (

                math.log10(
                    mt_aff
                )

                -

                math.log10(
                    wt_aff
                )
            )

        else:

            delta_log = None


        # ----------------------------------------------------------
        # Pair ligand checks
        # ----------------------------------------------------------

        wt_full_key = (
            wt.get(
                "Ligand_InChIKey"
            )
        )

        mt_full_key = (
            mt.get(
                "Ligand_InChIKey"
            )
        )


        same_full_ligand = (

            pd.notna(
                wt_full_key
            )

            and

            pd.notna(
                mt_full_key
            )

            and

            str(
                wt_full_key
            )

            ==

            str(
                mt_full_key
            )
        )


        # ----------------------------------------------------------
        # Strict QC
        # ----------------------------------------------------------

        strict_reasons = []


        # Exact numerical affinity
        if (
            wt.get(
                "Affinity_Relation"
            )
            != "="
        ):

            strict_reasons.append(
                "WT_affinity_not_exact"
            )


        if (
            mt.get(
                "Affinity_Relation"
            )
            != "="
        ):

            strict_reasons.append(
                "MT_affinity_not_exact"
            )


        if pd.isna(
            wt_aff
        ):

            strict_reasons.append(
                "WT_affinity_missing"
            )


        if pd.isna(
            mt_aff
        ):

            strict_reasons.append(
                "MT_affinity_missing"
            )


        # ligand identity
        if not same_full_ligand:

            strict_reasons.append(
                "full_InChIKey_diff_or_missing"
            )


        # alignment quality
        if (
            wt_ann[
                "Alignment_Identity"
            ] is None

            or

            wt_ann[
                "Alignment_Identity"
            ] < 0.90
        ):

            strict_reasons.append(
                "WT_alignment_identity_low"
            )


        if (
            mt_ann[
                "Alignment_Identity"
            ] is None

            or

            mt_ann[
                "Alignment_Identity"
            ] < 0.90
        ):

            strict_reasons.append(
                "MT_alignment_identity_low"
            )


        if mapped_overlap < 0.90:

            strict_reasons.append(
                "construct_region_overlap_low"
            )


        # Pair-specific mutations should be directly observable
        # in both structures' mapped sequence region.
        direct_diff_set = set(
            direct_diffs
        )


        if not (
            pair_specific
            <=
            direct_diff_set
        ):

            strict_reasons.append(
                "mutation_not_directly_comparable"
            )


        # PDBbind flags
        for prefix, rec in [

            ("WT", wt),
            ("MT", mt)
        ]:

            if bool(
                rec.get(
                    "Covalent_Flag"
                )
            ):

                strict_reasons.append(
                    f"{prefix}_covalent"
                )


            if bool(
                rec.get(
                    "Incomplete_Ligand_Flag"
                )
            ):

                strict_reasons.append(
                    f"{prefix}_incomplete_ligand"
                )


            if bool(
                rec.get(
                    "Assay_Structure_Mismatch_Flag"
                )
            ):

                strict_reasons.append(
                    f"{prefix}_structure_assay_mismatch"
                )


            if bool(
                rec.get(
                    "Nonstandard_Assay_Flag"
                )
            ):

                strict_reasons.append(
                    f"{prefix}_nonstandard_assay"
                )


        strict_pair = (
            len(
                strict_reasons
            ) == 0
        )


        # ----------------------------------------------------------
        # Human-readable ligand name
        # ----------------------------------------------------------

        wt_lig_name = (
            wt.get(
                "Ligand_Name"
            )
        )

        mt_lig_name = (
            mt.get(
                "Ligand_Name"
            )
        )


        if (
            pd.notna(
                wt_lig_name
            )
            and
            pd.notna(
                mt_lig_name
            )
            and
            str(
                wt_lig_name
            )
            ==
            str(
                mt_lig_name
            )
        ):

            ligand_name = (
                wt_lig_name
            )

        else:

            ligand_name = (
                wt_lig_name
                if pd.notna(
                    wt_lig_name
                )
                else mt_lig_name
            )


        candidate_rows.append({

            # GROUP
            "Group_ID":
                group_id,

            "UniProt_ID":
                wt[
                    "UniProt_ID"
                ],

            "Target_Name":
                wt.get(
                    "RCSB_Target_Name"
                ),

            "Ligand_Name":
                ligand_name,

            "Ligand_Group_Key":
                wt[
                    "Ligand_Group_Key"
                ],

            "Ligand_InChIKey_WT":
                wt.get(
                    "Ligand_InChIKey"
                ),

            "Ligand_InChIKey_MT":
                mt.get(
                    "Ligand_InChIKey"
                ),

            "Same_Full_InChIKey":
                same_full_ligand,

            "Ligand_SMILES_WT":
                wt.get(
                    "Ligand_SMILES"
                ),

            "Ligand_SMILES_MT":
                mt.get(
                    "Ligand_SMILES"
                ),

            "PDBbind_Ligand_Code_WT":
                wt.get(
                    "PDBbind_Ligand_Code"
                ),

            "PDBbind_Ligand_Code_MT":
                mt.get(
                    "PDBbind_Ligand_Code"
                ),


            # HARD GROUP VARIABLES
            "Affinity_Reference":
                wt[
                    "Affinity_Reference"
                ],

            "Reference_Key":
                wt[
                    "Reference_Key"
                ],

            "Affinity_Type":
                wt[
                    "Affinity_Type"
                ],


            # WT / REFERENCE
            "WT_PDB":
                wt[
                    "PDB_ID"
                ],

            "WT_Chain":
                wt.get(
                    "Single_Chain_ID"
                ),

            "WT_Affinity_Raw":
                wt.get(
                    "Affinity_Raw"
                ),

            "WT_Affinity_nM":
                wt_aff,

            "WT_pAffinity":
                wt.get(
                    "pAffinity"
                ),

            "WT_Sequence":
                wt.get(
                    "Sequence"
                ),

            "WT_Substitutions_vs_UniProt":
                wt.get(
                    "Substitutions_vs_UniProt"
                ),

            "WT_Background_Substitution_Count":
                wt.get(
                    "Substitution_Count_vs_UniProt"
                ),

            "WT_RCSB_Mutation_Count":
                wt.get(
                    "RCSB_Mutation_Count"
                ),

            "WT_Alignment_Identity":
                wt_ann[
                    "Alignment_Identity"
                ],

            "WT_Alignment_Coverage":
                wt_ann[
                    "Alignment_Coverage"
                ],


            # MUTATION
            "Mutation":
                mutation,

            "Pair_Mutation_Count":
                len(
                    pair_specific
                ),

            "Direct_WT_MT_Differences":
                ";".join(
                    sorted(
                        direct_diffs
                    )
                ),


            # MT
            "MT_PDB":
                mt[
                    "PDB_ID"
                ],

            "MT_Chain":
                mt.get(
                    "Single_Chain_ID"
                ),

            "MT_Affinity_Raw":
                mt.get(
                    "Affinity_Raw"
                ),

            "MT_Affinity_nM":
                mt_aff,

            "MT_pAffinity":
                mt.get(
                    "pAffinity"
                ),

            "MT_Sequence":
                mt.get(
                    "Sequence"
                ),

            "MT_Substitutions_vs_UniProt":
                mt.get(
                    "Substitutions_vs_UniProt"
                ),

            "MT_Substitution_Count_vs_UniProt":
                mt.get(
                    "Substitution_Count_vs_UniProt"
                ),

            "MT_RCSB_Mutation_Count":
                mt.get(
                    "RCSB_Mutation_Count"
                ),

            "MT_Alignment_Identity":
                mt_ann[
                    "Alignment_Identity"
                ],

            "MT_Alignment_Coverage":
                mt_ann[
                    "Alignment_Coverage"
                ],


            # COMPARABILITY
            "Canonical_Region_Overlap":
                mapped_overlap,

            "Delta_log10_Experimental":
                delta_log,


            # GUARANTEED GROUP RULES
            "Same_UniProt":
                True,

            "Same_Ligand_Group":
                True,

            "Same_Reference":
                True,

            "Same_Affinity_Type":
                True,


            # STRICT
            "Strict_Pair":
                strict_pair,

            "Strict_Exclusion_Reasons":
                ";".join(
                    strict_reasons
                ),
        })


pairs_df = pd.DataFrame(
    candidate_rows
)


# ======================================================================
# 28. REMOVE TRUE DUPLICATE PAIRS
# ======================================================================

if len(
    pairs_df
):

    pairs_df = (

        pairs_df
        .drop_duplicates(

            subset=[

                "UniProt_ID",
                "Ligand_Group_Key",
                "Reference_Key",
                "Affinity_Type",
                "WT_PDB",
                "MT_PDB",
                "Mutation",
            ]
        )

        .reset_index(
            drop=True
        )
    )


pairs_df[
    "Pair_ID"
] = [

    f"PAIR_{i:05d}"

    for i in range(
        1,
        len(pairs_df) + 1
    )
]


# Put Pair_ID first
pairs_df = pairs_df[
    [
        "Pair_ID"
    ]
    +
    [
        x
        for x in pairs_df.columns
        if x != "Pair_ID"
    ]
]


strict_df = (

    pairs_df[
        pairs_df[
            "Strict_Pair"
        ] == True
    ]

    .copy()

    .reset_index(
        drop=True
    )
)


print(
    "\nWT/MT candidate pairs:",
    len(pairs_df)
)

print(
    "Strict pairs:",
    len(strict_df)
)


# ======================================================================
# 29. ABSOLUTE GROUP VALIDATION
#
# No Group_ID may mix:
#
#   reference
#   Kd/Ki/IC50
#   UniProt
#   ligand
# ======================================================================

if len(
    pairs_df
):

    group_validation = (

        pairs_df
        .groupby(
            "Group_ID"
        )
        .agg(

            N_UniProt=(
                "UniProt_ID",
                "nunique"
            ),

            N_Ligands=(
                "Ligand_Group_Key",
                "nunique"
            ),

            N_References=(
                "Reference_Key",
                "nunique"
            ),

            N_Affinity_Types=(
                "Affinity_Type",
                "nunique"
            ),

            N_Pairs=(
                "Pair_ID",
                "nunique"
            ),
        )

        .reset_index()
    )


    group_validation[
        "Group_Valid"
    ] = (

        (
            group_validation[
                "N_UniProt"
            ] == 1
        )

        &

        (
            group_validation[
                "N_Ligands"
            ] == 1
        )

        &

        (
            group_validation[
                "N_References"
            ] == 1
        )

        &

        (
            group_validation[
                "N_Affinity_Types"
            ] == 1
        )
    )


    assert (
        group_validation[
            "Group_Valid"
        ].all()
    ), (
        "ERROR: mixed target/ligand/"
        "reference/endpoint group found."
    )

else:

    group_validation = (
        pd.DataFrame()
    )


# ======================================================================
# 30. GROUP SUMMARY
# ======================================================================

group_rows = []


if len(
    pairs_df
):

    for key, g in pairs_df.groupby(
        "Group_ID"
    ):

        wt_pdbs = sorted(
            set(
                g[
                    "WT_PDB"
                ]
            )
        )

        mt_pdbs = sorted(
            set(
                g[
                    "MT_PDB"
                ]
            )
        )

        mutations = sorted(
            set(
                g[
                    "Mutation"
                ]
            )
        )


        group_rows.append({

            "Group_ID":
                key,

            "UniProt_ID":
                g[
                    "UniProt_ID"
                ].iloc[0],

            "Target_Name":
                g[
                    "Target_Name"
                ].iloc[0],

            "Ligand_Name":
                g[
                    "Ligand_Name"
                ].iloc[0],

            "Ligand_Group_Key":
                g[
                    "Ligand_Group_Key"
                ].iloc[0],

            "Affinity_Reference":
                g[
                    "Affinity_Reference"
                ].iloc[0],

            "Affinity_Type":
                g[
                    "Affinity_Type"
                ].iloc[0],

            "N_WT_PDB":
                len(
                    wt_pdbs
                ),

            "WT_PDBs":
                ";".join(
                    wt_pdbs
                ),

            "N_MT_PDB":
                len(
                    mt_pdbs
                ),

            "MT_PDBs":
                ";".join(
                    mt_pdbs
                ),

            "Mutations":
                ";".join(
                    mutations
                ),

            "N_Pair_Comparisons":
                len(g),

            "N_Strict_Pairs":
                int(
                    g[
                        "Strict_Pair"
                    ].sum()
                ),
        })


groups_df = pd.DataFrame(
    group_rows
)


# ======================================================================
# 31. FLAT TABLE GENERATOR
# ======================================================================

def make_flat(
    pair_table
):

    rows = []


    for _, p in (
        pair_table.iterrows()
    ):

        # ----------------------------------------------------------
        # WT
        # ----------------------------------------------------------

        rows.append({

            "Pair_ID":
                p[
                    "Pair_ID"
                ],

            "Group_ID":
                p[
                    "Group_ID"
                ],

            "Variant_Status":
                "WT",

            "UniProt_ID":
                p[
                    "UniProt_ID"
                ],

            "target_name":
                p[
                    "Target_Name"
                ],

            "mutant":
                "WT",

            "pdb_id":
                p[
                    "WT_PDB"
                ],

            "ligand_name":
                p[
                    "Ligand_Name"
                ],

            "PDBbind_Ligand_Code":
                p[
                    "PDBbind_Ligand_Code_WT"
                ],

            "smiles":
                p[
                    "Ligand_SMILES_WT"
                ],

            "Ligand_InChIKey":
                p[
                    "Ligand_InChIKey_WT"
                ],

            "Ligand_Group_Key":
                p[
                    "Ligand_Group_Key"
                ],

            "sequence":
                p[
                    "WT_Sequence"
                ],

            "affinity":
                p[
                    "WT_Affinity_Raw"
                ],

            "affinity_nM":
                p[
                    "WT_Affinity_nM"
                ],

            "affinity_reference":
                p[
                    "Affinity_Reference"
                ],

            "affinity_type":
                p[
                    "Affinity_Type"
                ],

            "Pair_Specific_Mutations":
                p[
                    "Mutation"
                ],

            "Strict_Pair":
                p[
                    "Strict_Pair"
                ],
        })


        # ----------------------------------------------------------
        # MT
        # ----------------------------------------------------------

        rows.append({

            "Pair_ID":
                p[
                    "Pair_ID"
                ],

            "Group_ID":
                p[
                    "Group_ID"
                ],

            "Variant_Status":
                "MT",

            "UniProt_ID":
                p[
                    "UniProt_ID"
                ],

            "target_name":
                p[
                    "Target_Name"
                ],

            "mutant":
                p[
                    "Mutation"
                ],

            "pdb_id":
                p[
                    "MT_PDB"
                ],

            "ligand_name":
                p[
                    "Ligand_Name"
                ],

            "PDBbind_Ligand_Code":
                p[
                    "PDBbind_Ligand_Code_MT"
                ],

            "smiles":
                p[
                    "Ligand_SMILES_MT"
                ],

            "Ligand_InChIKey":
                p[
                    "Ligand_InChIKey_MT"
                ],

            "Ligand_Group_Key":
                p[
                    "Ligand_Group_Key"
                ],

            "sequence":
                p[
                    "MT_Sequence"
                ],

            "affinity":
                p[
                    "MT_Affinity_Raw"
                ],

            "affinity_nM":
                p[
                    "MT_Affinity_nM"
                ],

            "affinity_reference":
                p[
                    "Affinity_Reference"
                ],

            "affinity_type":
                p[
                    "Affinity_Type"
                ],

            "Pair_Specific_Mutations":
                p[
                    "Mutation"
                ],

            "Strict_Pair":
                p[
                    "Strict_Pair"
                ],
        })


    return pd.DataFrame(
        rows
    )


candidate_flat = make_flat(
    pairs_df
)

strict_flat = make_flat(
    strict_df
)


# ======================================================================
# 32. UNIQUE BOLTZ JOBS
#
# Same PDB + ligand + protein sequence calculated only once.
#
# If one WT is reference for 6 mutants, WT runs once.
# ======================================================================

BOLTZ_KEY = [

    "pdb_id",
    "Ligand_Group_Key",
    "sequence",
]


def make_unique_boltz(
    flat
):

    if len(flat) == 0:

        return pd.DataFrame()


    rows = []


    for key, g in flat.groupby(
        BOLTZ_KEY,
        dropna=False,
        sort=False
    ):

        first = (
            g.iloc[0]
        )


        rows.append({

            "pdb_id":
                first[
                    "pdb_id"
                ],

            "UniProt_ID":
                first[
                    "UniProt_ID"
                ],

            "target_name":
                first[
                    "target_name"
                ],

            "Variant_Status":
                first[
                    "Variant_Status"
                ],

            "mutant":
                first[
                    "mutant"
                ],

            "ligand_name":
                first[
                    "ligand_name"
                ],

            "PDBbind_Ligand_Code":
                first[
                    "PDBbind_Ligand_Code"
                ],

            "smiles":
                first[
                    "smiles"
                ],

            "Ligand_InChIKey":
                first[
                    "Ligand_InChIKey"
                ],

            "Ligand_Group_Key":
                first[
                    "Ligand_Group_Key"
                ],

            "sequence":
                first[
                    "sequence"
                ],

            "affinity":
                first[
                    "affinity"
                ],

            "affinity_nM":
                first[
                    "affinity_nM"
                ],

            "affinity_reference":
                first[
                    "affinity_reference"
                ],

            "affinity_type":
                first[
                    "affinity_type"
                ],

            "Pair_IDs":
                ";".join(
                    sorted(
                        set(
                            g[
                                "Pair_ID"
                            ]
                        )
                    )
                ),

            "Group_IDs":
                ";".join(
                    sorted(
                        set(
                            g[
                                "Group_ID"
                            ]
                        )
                    )
                ),
        })


    out = pd.DataFrame(
        rows
    )


    out.insert(

        0,

        "Boltz_Job_ID",

        [
            f"BOLTZ_{i:05d}"

            for i in range(
                1,
                len(out) + 1
            )
        ]
    )


    return out


boltz_unique = make_unique_boltz(
    candidate_flat
)

boltz_strict = make_unique_boltz(
    strict_flat
)


# ======================================================================
# 33. MAP BOLTZ JOB IDs BACK TO PAIRS
# ======================================================================

def build_job_lookup(
    jobs
):

    lookup = {}


    for _, r in jobs.iterrows():

        key = (

            r[
                "pdb_id"
            ],

            r[
                "Ligand_Group_Key"
            ],

            r[
                "sequence"
            ]
        )


        lookup[
            key
        ] = (
            r[
                "Boltz_Job_ID"
            ]
        )


    return lookup


candidate_job_lookup = (
    build_job_lookup(
        boltz_unique
    )
)


pair_map_rows = []


for _, p in (
    pairs_df.iterrows()
):

    wt_key = (

        p[
            "WT_PDB"
        ],

        p[
            "Ligand_Group_Key"
        ],

        p[
            "WT_Sequence"
        ]
    )


    mt_key = (

        p[
            "MT_PDB"
        ],

        p[
            "Ligand_Group_Key"
        ],

        p[
            "MT_Sequence"
        ]
    )


    pair_map_rows.append({

        "Pair_ID":
            p[
                "Pair_ID"
            ],

        "Group_ID":
            p[
                "Group_ID"
            ],

        "UniProt_ID":
            p[
                "UniProt_ID"
            ],

        "Target_Name":
            p[
                "Target_Name"
            ],

        "Ligand_Name":
            p[
                "Ligand_Name"
            ],

        "Affinity_Type":
            p[
                "Affinity_Type"
            ],

        "Affinity_Reference":
            p[
                "Affinity_Reference"
            ],

        "WT_PDB":
            p[
                "WT_PDB"
            ],

        "WT_Boltz_Job_ID":
            candidate_job_lookup.get(
                wt_key
            ),

        "Mutation":
            p[
                "Mutation"
            ],

        "MT_PDB":
            p[
                "MT_PDB"
            ],

        "MT_Boltz_Job_ID":
            candidate_job_lookup.get(
                mt_key
            ),

        "Delta_log10_Experimental":
            p[
                "Delta_log10_Experimental"
            ],

        "Strict_Pair":
            p[
                "Strict_Pair"
            ],
    })


pair_map_df = pd.DataFrame(
    pair_map_rows
)


# ======================================================================
# 34. SAVE CSVs
# ======================================================================

single_chain.to_csv(

    MASTER_CSV,

    index=False,

    compression="gzip"
)


boltz_unique.to_csv(

    BOLTZ_CANDIDATE_CSV,

    index=False
)


boltz_strict.to_csv(

    BOLTZ_STRICT_CSV,

    index=False
)


# ======================================================================
# 35. LIGAND NAME MAP
# ======================================================================

ligand_name_map = (

    single_chain[
        [
            "PDBbind_Ligand_Code",
            "Ligand_InChIKey",
            "Ligand_InChIKey14",
            "Ligand_Name",
            "Ligand_CCD_Name",
            "Ligand_PubChem_Name",
            "Ligand_Name_Source",
        ]
    ]

    .drop_duplicates()

    .sort_values(
        [
            "Ligand_Name",
            "PDBbind_Ligand_Code"
        ],
        na_position="last"
    )
)


# ======================================================================
# 36. SUMMARY
# ======================================================================

summary_df = pd.DataFrame({

    "Metric": [

        "PDBbind General rows",

        "Protein files found",

        "Exactly one protein chain",

        "Single-chain with exactly one UniProt",

        "Rows in potential multi-structure groups",

        "WT/MT candidate comparisons",

        "Strict WT/MT comparisons",

        "WT/MT groups",

        "Candidate flat rows",

        "Unique candidate Boltz jobs",

        "Strict flat rows",

        "Unique strict Boltz jobs",
    ],

    "Count": [

        len(
            index_df
        ),

        int(
            master[
                "Protein_File_Found"
            ]
            .fillna(False)
            .sum()
        ),

        len(
            single_chain
        ),

        int(
            (
                single_chain[
                    "UniProt_ID_Count"
                ]
                == 1
            ).sum()
        ),

        len(
            pair_input
        ),

        len(
            pairs_df
        ),

        len(
            strict_df
        ),

        len(
            groups_df
        ),

        len(
            candidate_flat
        ),

        len(
            boltz_unique
        ),

        len(
            strict_flat
        ),

        len(
            boltz_strict
        ),
    ]
})


display(
    summary_df
)


# ======================================================================
# 37. EXCEL-SAFE COPY
# ======================================================================

EXCEL_MAX = 32700


def excel_safe_value(x):

    if (
        isinstance(x, str)
        and
        len(x) > EXCEL_MAX
    ):

        return (
            x[:EXCEL_MAX]
            +
            "...[TRUNCATED_FOR_EXCEL]"
        )

    return x


def excel_safe_df(
    df
):

    out = df.copy()


    for col in out.columns:

        if out[
            col
        ].dtype == "object":

            out[
                col
            ] = (
                out[
                    col
                ]
                .map(
                    excel_safe_value
                )
            )


    return out


# ======================================================================
# 38. CHOOSE USEFUL MASTER COLUMNS
# ======================================================================

master_columns = [

    "PDB_ID",

    "UniProt_ID",
    "RCSB_Target_Name",

    "Protein_Chain_Count",
    "Single_Chain_ID",

    "Sequence",
    "Sequence_Length",

    "PDBbind_Ligand_Code",

    "Ligand_Name",
    "Ligand_CCD_Name",
    "Ligand_PubChem_Name",
    "Ligand_Name_Source",

    "Ligand_SMILES",
    "Ligand_InChIKey",
    "Ligand_InChIKey14",
    "Ligand_Group_Key",

    "Affinity_Raw",
    "Affinity_Type",
    "Affinity_Relation",
    "Affinity_nM",
    "pAffinity",

    "Affinity_Reference",
    "Reference_Key",

    "Resolution",
    "Release_Year",

    "RCSB_Mutation_Count",

    "Covalent_Flag",
    "Incomplete_Ligand_Flag",
    "Assay_Structure_Mismatch_Flag",
    "Nonstandard_Assay_Flag",

    "Index_Comment",

    "RCSB_Entity_Select_Status",
]


master_columns = [

    c

    for c in master_columns

    if c in single_chain.columns
]


# ======================================================================
# 39. WRITE XLSX
# ======================================================================

with pd.ExcelWriter(

    OUT_XLSX,

    engine="openpyxl"
) as writer:


    excel_safe_df(
        summary_df
    ).to_excel(

        writer,

        sheet_name="Summary",

        index=False
    )


    excel_safe_df(
        single_chain[
            master_columns
        ]
    ).to_excel(

        writer,

        sheet_name="SingleChain_Master",

        index=False
    )


    excel_safe_df(
        groups_df
    ).to_excel(

        writer,

        sheet_name="WT_MT_Groups",

        index=False
    )


    excel_safe_df(
        pairs_df
    ).to_excel(

        writer,

        sheet_name="WT_MT_Candidates",

        index=False
    )


    excel_safe_df(
        strict_df
    ).to_excel(

        writer,

        sheet_name="WT_MT_Strict",

        index=False
    )


    excel_safe_df(
        candidate_flat
    ).to_excel(

        writer,

        sheet_name="Candidate_Flat",

        index=False
    )


    excel_safe_df(
        strict_flat
    ).to_excel(

        writer,

        sheet_name="Strict_Flat",

        index=False
    )


    excel_safe_df(
        boltz_unique
    ).to_excel(

        writer,

        sheet_name="Boltz_Unique_Input",

        index=False
    )


    excel_safe_df(
        boltz_strict
    ).to_excel(

        writer,

        sheet_name="Boltz_Strict_Input",

        index=False
    )


    excel_safe_df(
        pair_map_df
    ).to_excel(

        writer,

        sheet_name="Boltz_Pair_Map",

        index=False
    )


    excel_safe_df(
        ligand_name_map
    ).to_excel(

        writer,

        sheet_name="Ligand_Name_Map",

        index=False
    )


# ======================================================================
# 40. FORMAT XLSX
# ======================================================================

wb = load_workbook(
    OUT_XLSX
)


for ws in wb.worksheets:

    ws.freeze_panes = "A2"


    if (
        ws.max_row > 1
        and
        ws.max_column > 0
    ):

        ws.auto_filter.ref = (
            ws.dimensions
        )


    for cell in ws[1]:

        cell.font = Font(
            bold=True
        )

        cell.alignment = Alignment(
            wrap_text=True,
            vertical="top"
        )


    for col_idx in range(
        1,
        ws.max_column + 1
    ):

        header = str(
            ws.cell(
                1,
                col_idx
            ).value
            or ""
        ).lower()


        if "sequence" in header:

            width = 45

        elif "smiles" in header:

            width = 45

        elif "ligand_name" in header:

            width = 35

        elif "comment" in header:

            width = 50

        elif "exclusion" in header:

            width = 45

        elif "mutation" in header:

            width = 30

        elif "target" in header:

            width = 32

        elif "pdb" in header:

            width = 14

        else:

            width = 18


        ws.column_dimensions[
            get_column_letter(
                col_idx
            )
        ].width = width


    for row in ws.iter_rows():

        for cell in row:

            cell.alignment = Alignment(
                wrap_text=True,
                vertical="top"
            )


wb.save(
    OUT_XLSX
)


# ======================================================================
# 41. FINAL VALIDATION
# ======================================================================

print("\n" + "="*80)
print("FINAL VALIDATION")
print("="*80)


if len(
    pairs_df
):

    assert (
        pairs_df[
            "Same_UniProt"
        ].all()
    )

    assert (
        pairs_df[
            "Same_Ligand_Group"
        ].all()
    )

    assert (
        pairs_df[
            "Same_Reference"
        ].all()
    )

    assert (
        pairs_df[
            "Same_Affinity_Type"
        ].all()
    )


    assert not pairs_df.duplicated(

        subset=[

            "UniProt_ID",
            "Ligand_Group_Key",
            "Reference_Key",
            "Affinity_Type",
            "WT_PDB",
            "MT_PDB",
            "Mutation",
        ]

    ).any()


print(
    "✓ Every structure has exactly 1 protein chain"
)

print(
    "✓ Every pair has same UniProt"
)

print(
    "✓ Every pair has same ligand grouping identity"
)

print(
    "✓ Every pair has same affinity reference"
)

print(
    "✓ Every pair has same Kd/Ki/IC50 endpoint"
)

print(
    "✓ Pair mutation names derived from WT/MT sequence comparison"
)

print(
    "✓ Boltz input deduplicated by PDB + ligand + sequence"
)


print("\nOUTPUT FILES:")

print(
    OUT_XLSX
)

print(
    MASTER_CSV
)

print(
    BOLTZ_CANDIDATE_CSV
)

print(
    BOLTZ_STRICT_CSV
)


print("\nSUMMARY:")

display(
    summary_df
)


# ======================================================================
# 42. PREVIEW WT/MT PAIRS
# ======================================================================

if len(
    pairs_df
):

    preview_cols = [

        "Pair_ID",
        "Group_ID",
        "UniProt_ID",
        "Target_Name",
        "Ligand_Name",
        "Affinity_Type",
        "Affinity_Reference",

        "WT_PDB",
        "WT_Affinity_Raw",

        "WT_Substitutions_vs_UniProt",

        "Mutation",

        "MT_PDB",
        "MT_Affinity_Raw",

        "MT_Substitutions_vs_UniProt",

        "Delta_log10_Experimental",

        "Strict_Pair",
        "Strict_Exclusion_Reasons",
    ]


    display(
        pairs_df[
            preview_cols
        ].head(50)
    )


# ======================================================================
# 43. OPTIONAL DOWNLOAD
# ======================================================================

#
# from google.colab import files
# files.download(str(OUT_XLSX))